# Part 1 — Representative-cell reduction with K-medoids

This notebook reduces the number of cells in a **gene × cell expression matrix** while retaining actual observed cells as representatives (medoids).

### Input
`ExpressionData.csv` with genes in rows and cells in columns. The first column contains gene names.

### Output
- reduced expression matrix containing only selected medoid cells;
- selected-cell metadata;
- cluster-quality summary;
- optional random-selection baseline using the same number of cells.

The notebook is deliberately dataset-agnostic. Edit only the **Configuration** cell for a new dataset.


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.preprocessing import normalize

# Run this notebook either from the repository root or from notebooks/.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
print("Project root:", PROJECT_ROOT)


## Configuration

For large datasets, the pairwise cell-distance matrix requires approximately \(O(n_{cells}^2)\) memory. Start with a smaller subset if memory is limited.


In [ ]:
# ---------- USER SETTINGS ----------
INPUT_CSV = PROJECT_ROOT / "data/example/ExpressionData.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs/part1"

K = 30
DISTANCE_METRIC = "cosine"   # one of: corr_abs, corr_signed, cosine, euclidean
DO_LOG1P = True
ZSCORE_GENES = True
GENE_KEEP_QUANTILE = 1.0      # 1.0 keeps all genes
RANDOM_SEED = 42
N_INIT = 3
MAX_ITER = 50
COMPUTE_SILHOUETTE = True
WRITE_RANDOM_BASELINE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Load and validate the expression matrix


In [ ]:
def read_expression_csv(path):
    df = pd.read_csv(path, index_col=0)
    if df.empty:
        raise ValueError("Expression matrix is empty.")
    df = df.apply(pd.to_numeric, errors="coerce")
    if df.isna().any().any():
        raise ValueError("Expression matrix contains missing/non-numeric values after parsing.")
    if df.columns.duplicated().any():
        raise ValueError("Cell names must be unique.")
    if df.index.duplicated().any():
        raise ValueError("Gene names must be unique.")
    return df

df = read_expression_csv(INPUT_CSV)
print(f"Loaded {df.shape[0]} genes × {df.shape[1]} cells")
display(df.iloc[:5, :6])


## Distance preprocessing and K-medoids


In [ ]:
def preprocess_for_distance(df, do_log1p=True, zscore_genes=True, gene_keep_quantile=1.0):
    X = df.to_numpy(dtype=float).copy()  # genes × cells
    if np.any(X < 0) and do_log1p:
        raise ValueError("log1p was requested but negative expression values are present.")
    if do_log1p:
        X = np.log1p(X)

    # Optional removal of the least-variable genes for distance calculation only.
    if gene_keep_quantile < 1.0:
        variances = np.var(X, axis=1)
        cutoff = np.quantile(variances, 1.0 - gene_keep_quantile)
        keep = variances >= cutoff
        X = X[keep]

    if zscore_genes:
        mu = X.mean(axis=1, keepdims=True)
        sd = X.std(axis=1, keepdims=True)
        sd[sd < 1e-12] = 1.0
        X = (X - mu) / sd

    return X.T  # cells × genes


def column_distance_matrix(X_cells_by_genes, metric="cosine"):
    if metric == "corr_abs":
        corr = np.corrcoef(X_cells_by_genes)
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
        D = 1.0 - np.abs(corr)
    elif metric == "corr_signed":
        corr = np.corrcoef(X_cells_by_genes)
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
        D = 1.0 - ((corr + 1.0) / 2.0)
    elif metric == "cosine":
        Xn = normalize(X_cells_by_genes)
        sim = np.clip(Xn @ Xn.T, -1.0, 1.0)
        D = 1.0 - ((sim + 1.0) / 2.0)
    elif metric == "euclidean":
        D = pairwise_distances(X_cells_by_genes, metric="euclidean")
    else:
        raise ValueError("DISTANCE_METRIC must be corr_abs, corr_signed, cosine, or euclidean")
    D = np.asarray(D, dtype=float)
    D = np.nan_to_num(D, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(D, 0.0)
    return D


def farthest_first_init(D, k, rng):
    n = D.shape[0]
    first = int(rng.integers(0, n))
    medoids = [first]
    min_dist = D[:, first].copy()
    while len(medoids) < k:
        next_idx = int(np.argmax(min_dist))
        if next_idx in medoids:
            remaining = np.setdiff1d(np.arange(n), np.asarray(medoids))
            next_idx = int(rng.choice(remaining))
        medoids.append(next_idx)
        min_dist = np.minimum(min_dist, D[:, next_idx])
    return np.asarray(medoids, dtype=int)


def assign_to_medoids(D, medoids):
    dist = D[:, medoids]
    labels = np.argmin(dist, axis=1)
    min_distances = dist[np.arange(D.shape[0]), labels]
    return labels, min_distances, float(min_distances.sum())


def k_medoids(D, k, seed=42, n_init=3, max_iter=50):
    n = D.shape[0]
    if not 1 <= k <= n:
        raise ValueError(f"k must satisfy 1 <= k <= number of cells ({n}).")
    if k == n:
        medoids = np.arange(n)
        labels, min_dist, cost = assign_to_medoids(D, medoids)
        return medoids, labels, min_dist, cost, 0

    best = None
    for init_id in range(n_init):
        rng = np.random.default_rng(seed + init_id)
        medoids = farthest_first_init(D, k, rng)
        iterations = 0
        for iterations in range(1, max_iter + 1):
            labels, min_dist, cost = assign_to_medoids(D, medoids)
            new_medoids = medoids.copy()
            for cluster_id in range(k):
                members = np.where(labels == cluster_id)[0]
                if len(members) == 0:
                    remaining = np.setdiff1d(np.arange(n), new_medoids)
                    if len(remaining):
                        new_medoids[cluster_id] = int(rng.choice(remaining))
                    continue
                subD = D[np.ix_(members, members)]
                new_medoids[cluster_id] = int(members[np.argmin(subD.sum(axis=1))])
            if np.array_equal(np.sort(medoids), np.sort(new_medoids)):
                medoids = new_medoids
                break
            medoids = new_medoids

        labels, min_dist, cost = assign_to_medoids(D, medoids)
        candidate = (medoids.copy(), labels.copy(), min_dist.copy(), cost, iterations)
        if best is None or cost < best[3]:
            best = candidate
    return best


## Run cell reduction


In [ ]:
start = time.time()
Xp = preprocess_for_distance(df, DO_LOG1P, ZSCORE_GENES, GENE_KEEP_QUANTILE)
D = column_distance_matrix(Xp, DISTANCE_METRIC)
medoids, labels, min_distances, total_cost, iterations = k_medoids(
    D, K, seed=RANDOM_SEED, n_init=N_INIT, max_iter=MAX_ITER
)

# Save reduced matrix in original cell-column order.
medoids_sorted = np.sort(medoids)
selected_cells = df.columns[medoids_sorted]
reduced = df.loc[:, selected_cells]
reduced_file = OUTPUT_DIR / f"ExpressionData_kmedoids_k{K}_{DISTANCE_METRIC}.csv"
reduced.to_csv(reduced_file)

rows=[]
for cluster_id, medoid_idx in enumerate(medoids):
    members=np.where(labels==cluster_id)[0]
    rows.append({
        "ClusterID": cluster_id,
        "SelectedCell": df.columns[medoid_idx],
        "OriginalColumnIndex": int(medoid_idx),
        "ClusterSize": int(len(members)),
        "MeanDistanceWithinCluster": float(D[members, medoid_idx].mean()),
        "MaxDistanceWithinCluster": float(D[members, medoid_idx].max()),
    })
selected_df=pd.DataFrame(rows).sort_values("OriginalColumnIndex")
selected_file=OUTPUT_DIR / f"selected_cells_k{K}_{DISTANCE_METRIC}.csv"
selected_df.to_csv(selected_file,index=False)

summary={
    "OriginalNumCells": int(df.shape[1]),
    "ReducedNumCells": int(K),
    "ReductionRatio": float(K/df.shape[1]),
    "DistanceMetric": DISTANCE_METRIC,
    "DoLog1p": DO_LOG1P,
    "ZScoreGenes": ZSCORE_GENES,
    "TotalWithinClusterDistance": float(total_cost),
    "MeanWithinClusterDistance": float(np.mean(min_distances)),
    "MedianWithinClusterDistance": float(np.median(min_distances)),
    "MaxWithinClusterDistance": float(np.max(min_distances)),
    "Iterations": int(iterations),
    "RuntimeSeconds": float(time.time()-start),
}
if COMPUTE_SILHOUETTE and len(np.unique(labels)) > 1 and len(np.unique(labels)) < len(labels):
    summary["SilhouettePrecomputed"] = float(silhouette_score(D, labels, metric="precomputed"))

summary_file=OUTPUT_DIR / f"quality_k{K}_{DISTANCE_METRIC}.csv"
pd.DataFrame([summary]).to_csv(summary_file,index=False)

print("Reduced matrix:", reduced_file)
print("Selected cells:", selected_file)
print("Quality summary:", summary_file)
display(pd.DataFrame([summary]))


## Optional matched-size random baseline


In [ ]:
if WRITE_RANDOM_BASELINE:
    rng=np.random.default_rng(RANDOM_SEED)
    random_idx=np.sort(rng.choice(df.shape[1], size=K, replace=False))
    random_cells=df.columns[random_idx]
    random_df=df.loc[:, random_cells]
    random_file=OUTPUT_DIR / f"ExpressionData_random_k{K}_seed{RANDOM_SEED}.csv"
    random_df.to_csv(random_file)
    pd.DataFrame({
        "SelectedCell": random_cells,
        "OriginalColumnIndex": random_idx,
    }).to_csv(OUTPUT_DIR / f"selected_cells_random_k{K}_seed{RANDOM_SEED}.csv", index=False)
    print("Random baseline:", random_file)


## Next step

Run each GRN inference algorithm on the same reduced expression matrix. Save each prediction as a CSV with three columns:

```text
Gene1,Gene2,EdgeWeight
G01,G02,0.91
G01,G03,0.72
...
```

`EdgeWeight` only needs to be **ordinal within a method**: larger values must mean stronger inferred evidence. The raw values do not need to be calibrated probabilities.

Then use **Part 2** to fuse two or more GRN prediction files.
